# Power Outage Severity by Location

**Name(s)**: Andrew Zaletski

**Website Link**: azaletski24.github.io/Andrew-Zaletski/

In [85]:
import pandas as pd
import numpy as np
from pathlib import Path

from dsc80_utils import *

import plotly.express as px
pd.options.plotting.backend = 'plotly'

## Step 1: Introduction

In [86]:
"""
Step 1: Introduction and question identification.

In this project I use the "Major Power Outage Risks in the U.S." dataset, which
contains records of large power outages in the continental U.S. from 2000–2016.
My main question is location-based:

    Are major power outages more severe, in terms of the number of customers
    affected, in some parts of the U.S. than in others?

In particular, I will compare outages that occur in the **South** climate region
with outages that occur in the **West North Central** climate region, using the
`CLIMATE.REGION` and `CUSTOMERS.AFFECTED` columns.
"""

# Load the raw outage data that was exported from the original Excel file.
# We read it as an Excel workbook because the file is really an .xlsx file
# that happens to have a .csv extension.
data_path = Path("outage.xsxl.csv")
raw_outages = pd.read_excel(data_path, engine="openpyxl", header=None)

# Show the raw shape so we know how many rows/columns we start with.
raw_outages.shape

(1541, 57)

## Step 2: Data Cleaning and Exploratory Data Analysis

In [87]:
"""
Step 2: Data cleaning and exploratory data analysis.

Cleaning steps:
- Remove the non-data rows at the top of the sheet (title, description, etc.).
- Use row 5 (index 5) as the source of variable names and drop the first
  "variables" column, which is just a label.
- Drop the units row (index 6) and any completely empty rows.
- Combine the outage date and time columns into `OUTAGE.START` and
  `OUTAGE.RESTORATION` `Timestamp` columns.
- Coerce key numerical columns to numeric dtypes so that plotting/aggregation
  behave as expected.
"""

# Extract column names from row 5 (index 5), skipping the first "variables" cell.
header = raw_outages.iloc[5, 1:]

# Data starts at row 7 (index 7); again, skip the first column.
outages = raw_outages.iloc[7:, 1:].copy()
outages.columns = header

# Drop any rows that are entirely empty and reset the index.
outages = outages.dropna(how="all").reset_index(drop=True)

# Combine outage start date/time and restoration date/time into Timestamp columns.
start_date = pd.to_datetime(outages["OUTAGE.START.DATE"])
start_time = pd.to_timedelta(outages["OUTAGE.START.TIME"].astype(str))
rest_date = pd.to_datetime(outages["OUTAGE.RESTORATION.DATE"])
rest_time = pd.to_timedelta(outages["OUTAGE.RESTORATION.TIME"].astype(str))

outages["OUTAGE.START"] = start_date + start_time
outages["OUTAGE.RESTORATION"] = rest_date + rest_time

# Drop the original date/time columns now that we've combined them.
outages = outages.drop(
    columns=[
        "OUTAGE.START.DATE",
        "OUTAGE.START.TIME",
        "OUTAGE.RESTORATION.DATE",
        "OUTAGE.RESTORATION.TIME",
    ]
)

# Coerce key numerical columns to numeric types (invalid entries become NaN).
numeric_cols = [
    "OUTAGE.DURATION",
    "DEMAND.LOSS.MW",
    "CUSTOMERS.AFFECTED",
    "TOTAL.PRICE",
    "TOTAL.CUSTOMERS",
]
for col in numeric_cols:
    if col in outages.columns:
        outages[col] = pd.to_numeric(outages[col], errors="coerce")

print("Cleaned outages shape:", outages.shape)
outages.head()

# Step 2 visualization: number of major power outages by U.S. climate region (univariate).
region_counts = (
    outages["CLIMATE.REGION"]
    .value_counts()
    .reset_index()
    .rename(columns={"CLIMATE.REGION": "CLIMATE.REGION", "count": "num_outages"})
)
fig_region = px.bar(
    region_counts,
    x="CLIMATE.REGION",
    y="num_outages",
    title="Number of Major Power Outages by U.S. Climate Region",
    labels={"CLIMATE.REGION": "Climate Region", "num_outages": "Number of Outages"},
)
fig_region.update_layout(xaxis_tickangle=-30)
fig_region

Cleaned outages shape: (1534, 54)


In [88]:
"""
Additional Step 2 EDA: more univariate, bivariate, and aggregate analyses.
"""

# Univariate: distribution of customers affected (log scale on x-axis).
fig_customers = px.histogram(
    outages,
    x="CUSTOMERS.AFFECTED",
    nbins=40,
    title="Distribution of Customers Affected per Outage",
    labels={"CUSTOMERS.AFFECTED": "Customers Affected"},
)
fig_customers.update_xaxes(type="log")
fig_customers

# Bivariate: customers affected by climate region (box plot).
fig_box = px.box(
    outages,
    x="CLIMATE.REGION",
    y="CUSTOMERS.AFFECTED",
    title="Customers Affected by Climate Region",
    labels={"CLIMATE.REGION": "Climate Region", "CUSTOMERS.AFFECTED": "Customers Affected"},
)
fig_box.update_xaxes(tickangle=-30)
fig_box

# Bivariate: outage duration vs customers affected.
fig_scatter = px.scatter(
    outages,
    x="OUTAGE.DURATION",
    y="CUSTOMERS.AFFECTED",
    title="Outage Duration vs. Customers Affected",
    labels={"OUTAGE.DURATION": "Outage Duration (minutes)", "CUSTOMERS.AFFECTED": "Customers Affected"},
)
fig_scatter

# Interesting aggregate: mean total price and mean total customers per state.
state_agg = (
    outages.groupby("U.S._STATE")[["TOTAL.PRICE", "TOTAL.CUSTOMERS"]]
    .mean()
    .reset_index()
    .rename(columns={"TOTAL.PRICE": "mean_total_price", "TOTAL.CUSTOMERS": "mean_total_customers"})
)

state_agg.head()

5,U.S._STATE,mean_total_price,mean_total_customers
0,Alabama,7.99,2.40e+06
1,Alaska,NaN,2.74e+05
2,Arizona,9.12,2.80e+06
3,Arkansas,7.62,1.54e+06
4,California,13.09,1.47e+07


## Step 3: Assessment of Missingness

In [89]:
# Column with non-trivial missingness: CUSTOMERS.AFFECTED
outages["CUSTOMERS.AFFECTED_missing"] = outages["CUSTOMERS.AFFECTED"].isna()

def perm_missingness_stat(df, miss_col, other_col):
    tbl = pd.crosstab(df[other_col], df[miss_col])
    if tbl.shape[1] < 2:
        return 0.0
    from scipy.stats import chi2_contingency
    chi2, _, _, _ = chi2_contingency(tbl)
    return chi2

n_reps = 500
miss = outages["CUSTOMERS.AFFECTED_missing"].values

# Test: does missingness of CUSTOMERS.AFFECTED depend on U.S._STATE?
obs_state = perm_missingness_stat(outages, "CUSTOMERS.AFFECTED_missing", "U.S._STATE")
perm_stats_state = []
for _ in range(n_reps):
    shuffled = np.random.permutation(miss)
    df_perm = outages.copy()
    df_perm["CUSTOMERS.AFFECTED_missing"] = shuffled
    perm_stats_state.append(perm_missingness_stat(df_perm, "CUSTOMERS.AFFECTED_missing", "U.S._STATE"))
p_state = np.mean(np.array(perm_stats_state) >= obs_state)
print("Missingness of CUSTOMERS.AFFECTED vs U.S._STATE: observed chi2 =", round(obs_state, 4), ", p-value =", p_state)

# Test: does missingness of CUSTOMERS.AFFECTED depend on CLIMATE.CATEGORY?
obs_clim = perm_missingness_stat(outages, "CUSTOMERS.AFFECTED_missing", "CLIMATE.CATEGORY")
perm_stats_clim = []
for _ in range(n_reps):
    shuffled = np.random.permutation(miss)
    df_perm = outages.copy()
    df_perm["CUSTOMERS.AFFECTED_missing"] = shuffled
    perm_stats_clim.append(perm_missingness_stat(df_perm, "CUSTOMERS.AFFECTED_missing", "CLIMATE.CATEGORY"))
p_clim = np.mean(np.array(perm_stats_clim) >= obs_clim)
print("Missingness of CUSTOMERS.AFFECTED vs CLIMATE.CATEGORY: observed chi2 =", round(obs_clim, 4), ", p-value =", p_clim)

# Plot: distribution of CLIMATE.REGION by missing vs not
fig_miss = px.histogram(
    outages.assign(missing=outages["CUSTOMERS.AFFECTED_missing"].map({True: "Missing", False: "Not missing"})),
    x="CLIMATE.REGION",
    color="missing",
    barmode="group",
    title="Distribution of CLIMATE.REGION by whether CUSTOMERS.AFFECTED is missing",
)
fig_miss.update_layout(xaxis_tickangle=-30)
fig_miss

Missingness of CUSTOMERS.AFFECTED vs U.S._STATE: observed chi2 = 242.5045 , p-value = 0.0
Missingness of CUSTOMERS.AFFECTED vs CLIMATE.CATEGORY: observed chi2 = 2.5675 , p-value = 0.288


## Step 4: Hypothesis Testing

In [90]:
# H0: Mean CUSTOMERS.AFFECTED is the same in South and West North Central.
# HA: Mean CUSTOMERS.AFFECTED is higher in South than in West North Central.
# Test statistic: mean(South) - mean(West North Central); one-sided.

df_ht = outages[outages["CLIMATE.REGION"].isin(["South", "West North Central"])].copy()
df_ht = df_ht.dropna(subset=["CUSTOMERS.AFFECTED"])
south = df_ht.loc[df_ht["CLIMATE.REGION"] == "South", "CUSTOMERS.AFFECTED"]
wnc = df_ht.loc[df_ht["CLIMATE.REGION"] == "West North Central", "CUSTOMERS.AFFECTED"]
observed = south.mean() - wnc.mean()

n_perm = 5000
perm_stats = []
for _ in range(n_perm):
    labels = np.random.permutation(df_ht["CLIMATE.REGION"].values)
    m_s = df_ht.loc[labels == "South", "CUSTOMERS.AFFECTED"].mean()
    m_w = df_ht.loc[labels == "West North Central", "CUSTOMERS.AFFECTED"].mean()
    perm_stats.append(m_s - m_w)
perm_stats = np.array(perm_stats)
p_value_ht = np.mean(perm_stats >= observed)
print("Observed difference (South - West North Central) mean CUSTOMERS.AFFECTED:", observed)
print("p-value (one-sided):", p_value_ht)
print("Conclusion: Reject H0 at 0.05" if p_value_ht < 0.05 else "Conclusion: Do not reject H0 at 0.05")

Observed difference (South - West North Central) mean CUSTOMERS.AFFECTED: 136184.7741935484
p-value (one-sided): 0.0376
Conclusion: Reject H0 at 0.05


## Step 5: Framing a Prediction Problem

In [91]:
# Prediction problem: regression. Response: CUSTOMERS.AFFECTED (number of customers affected).
# Metric: RMSE (root mean squared error). We use RMSE so predictions are in the same units as the target.
# Features known at "time of prediction" (e.g. at start of outage or when planning): CLIMATE.REGION,
# YEAR, MONTH, U.S._STATE, NERC.REGION, CAUSE.CATEGORY (if known), OUTAGE.DURATION (if known by the time we predict impact).
# We do not use future-only or outcome-derived columns.
# Train/test split: fixed random_state for reproducibility; same split used for baseline and final model.
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

outages_ml = outages.dropna(subset=["CUSTOMERS.AFFECTED"]).copy()
X_full = outages_ml[["YEAR", "CLIMATE.REGION"]].copy()
y_full = outages_ml["CUSTOMERS.AFFECTED"]
train_idx, test_idx = train_test_split(outages_ml.index, test_size=0.25, random_state=42)
train_df = outages_ml.loc[train_idx]
test_df = outages_ml.loc[test_idx]
y_train = y_full.loc[train_idx]
y_test = y_full.loc[test_idx]
X_train = train_df[["YEAR", "CLIMATE.REGION"]]
X_test = test_df[["YEAR", "CLIMATE.REGION"]]
print("Train size:", len(y_train), "Test size:", len(y_test))

Train size: 818 Test size: 273


## Step 6: Baseline Model

In [92]:
# Recompute fig_customers using only valid, positive values so the
# distribution plot is non-empty and matches the website embed.

customers_nonnull = outages.dropna(subset=["CUSTOMERS.AFFECTED"])
customers_nonnull = customers_nonnull[customers_nonnull["CUSTOMERS.AFFECTED"] > 0]

fig_customers = px.histogram(
    customers_nonnull,
    x="CUSTOMERS.AFFECTED",
    nbins=40,
    title="Distribution of Customers Affected per Outage",
    labels={"CUSTOMERS.AFFECTED": "Customers Affected"},
)
fig_customers.update_xaxes(type="log")
fig_customers

In [93]:
# Baseline: at least 2 features. YEAR (quantitative), CLIMATE.REGION (categorical). Pipeline with OneHotEncoder + LinearRegression.
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

ct = ColumnTransformer(
    [("num", "passthrough", ["YEAR"]), ("cat", OneHotEncoder(handle_unknown="ignore"), ["CLIMATE.REGION"])],
    remainder="drop",
)
pipe_baseline = Pipeline([("preprocess", ct), ("model", LinearRegression())])
pipe_baseline.fit(X_train, y_train)

rmse_train_baseline = np.sqrt(mean_squared_error(y_train, pipe_baseline.predict(X_train)))
rmse_test_baseline = np.sqrt(mean_squared_error(y_test, pipe_baseline.predict(X_test)))
print("Baseline RMSE (train):", rmse_train_baseline)
print("Baseline RMSE (test):", rmse_test_baseline)

baseline_train_pred = pipe_baseline.predict(X_train)
baseline_test_pred = pipe_baseline.predict(X_test)

fig_baseline_scatter = px.scatter(
    x=y_test,
    y=baseline_test_pred,
    title="Baseline Model: Actual vs Predicted (Test Set)",
    labels={"x": "Actual CUSTOMERS.AFFECTED", "y": "Predicted CUSTOMERS.AFFECTED"},
)
fig_baseline_scatter.add_shape(
    type="line",
    x0=y_test.min(), y0=y_test.min(),
    x1=y_test.max(), y1=y_test.max(),
    line=dict(color="red", dash="dash"),
)
fig_baseline_scatter

baseline_resid = y_test - baseline_test_pred
fig_baseline_resid = px.histogram(
    x=baseline_resid,
    nbins=40,
    title="Baseline Model: Residuals (Test Set)",
    labels={"x": "Residual (Actual - Predicted)", "y": "Count"},
)
fig_baseline_resid

Baseline RMSE (train): 254179.8031999797
Baseline RMSE (test): 343652.0477566325


## Step 7: Final Model

In [94]:
# Final model: add MONTH and OUTAGE.DURATION as new features; tune Ridge alpha via GridSearchCV.
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

X_train_final = train_df[["YEAR", "MONTH", "OUTAGE.DURATION", "CLIMATE.REGION"]].copy()
X_test_final = test_df[["YEAR", "MONTH", "OUTAGE.DURATION", "CLIMATE.REGION"]].copy()
for col in ["YEAR", "MONTH", "OUTAGE.DURATION"]:
    X_train_final[col] = pd.to_numeric(X_train_final[col], errors="coerce")
    X_test_final[col] = pd.to_numeric(X_test_final[col], errors="coerce")
X_train_final = X_train_final.fillna(X_train_final.median(numeric_only=True))
X_test_final = X_test_final.fillna(X_train_final.median(numeric_only=True))

ct_final = ColumnTransformer(
    [
        ("num", StandardScaler(), ["YEAR", "MONTH", "OUTAGE.DURATION"]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["CLIMATE.REGION"]),
    ],
    remainder="drop",
)
pipe_final = Pipeline([("preprocess", ct_final), ("model", Ridge())])
param_grid = {"model__alpha": [0.01, 0.1, 1.0, 10.0]}
grid = GridSearchCV(pipe_final, param_grid, cv=5, scoring="neg_root_mean_squared_error")
grid.fit(X_train_final, y_train)
pipe_final_best = grid.best_estimator_
rmse_train_final = np.sqrt(mean_squared_error(y_train, pipe_final_best.predict(X_train_final)))
rmse_test_final = np.sqrt(mean_squared_error(y_test, pipe_final_best.predict(X_test_final)))
print("Best alpha:", grid.best_params_)
print("Final RMSE (train):", rmse_train_final)
print("Final RMSE (test):", rmse_test_final)
print("Improvement over baseline test RMSE:", rmse_test_baseline - rmse_test_final)

final_test_pred = pipe_final_best.predict(X_test_final)

fig_final_scatter = px.scatter(
    x=y_test,
    y=final_test_pred,
    title="Final Model: Actual vs Predicted (Test Set)",
    labels={"x": "Actual CUSTOMERS.AFFECTED", "y": "Predicted CUSTOMERS.AFFECTED"},
)
fig_final_scatter.add_shape(
    type="line",
    x0=y_test.min(), y0=y_test.min(),
    x1=y_test.max(), y1=y_test.max(),
    line=dict(color="red", dash="dash"),
)
fig_final_scatter

final_resid = y_test - final_test_pred
fig_final_resid = px.histogram(
    x=final_resid,
    nbins=40,
    title="Final Model: Residuals (Test Set)",
    labels={"x": "Residual (Actual - Predicted)", "y": "Count"},
)
fig_final_resid

Best alpha: {'model__alpha': 10.0}
Final RMSE (train): 251105.904293663
Final RMSE (test): 329890.7621752107
Improvement over baseline test RMSE: 13761.285581421806


## Step 8: Fairness Analysis

In [95]:
# Fairness: does the final model perform worse for one climate region than another?
# Groups: South vs West North Central. Metric: RMSE within each group (on test set).
test_df_final = test_df.copy()
test_df_final["pred"] = pipe_final_best.predict(X_test_final)
south_test = test_df_final[test_df_final["CLIMATE.REGION"] == "South"]
wnc_test = test_df_final[test_df_final["CLIMATE.REGION"] == "West North Central"]
rmse_south = np.sqrt(mean_squared_error(south_test["CUSTOMERS.AFFECTED"], south_test["pred"]))
rmse_wnc = np.sqrt(mean_squared_error(wnc_test["CUSTOMERS.AFFECTED"], wnc_test["pred"]))
print("RMSE South:", rmse_south, "RMSE West North Central:", rmse_wnc)
# Permutation test: H0 model is fair (RMSE same); HA difference in RMSE. Statistic: |RMSE_South - RMSE_WNC|.
obs_fair = abs(rmse_south - rmse_wnc)
regions = test_df_final["CLIMATE.REGION"].values
y_te = test_df_final["CUSTOMERS.AFFECTED"].values
preds = test_df_final["pred"].values
n_fair = 1000
fair_stats = []
for _ in range(n_fair):
    perm_reg = np.random.permutation(regions)
    mask_s = perm_reg == "South"
    mask_w = perm_reg == "West North Central"
    if mask_s.sum() > 0 and mask_w.sum() > 0:
        r_s = np.sqrt(np.mean((y_te[mask_s] - preds[mask_s]) ** 2))
        r_w = np.sqrt(np.mean((y_te[mask_w] - preds[mask_w]) ** 2))
        fair_stats.append(abs(r_s - r_w))
fair_stats = np.array(fair_stats)
p_fair = np.mean(fair_stats >= obs_fair)
print("Observed |RMSE_South - RMSE_WNC|:", obs_fair, "p-value:", p_fair)

RMSE South: 597312.3809199072 RMSE West North Central: 21873.874044854045
Observed |RMSE_South - RMSE_WNC|: 575438.5068750532 p-value: 0.045


In [96]:
# Step 7: Feature importance for the final Ridge model.
# We look at the absolute value of each coefficient after preprocessing.

preprocessor = pipe_final_best.named_steps["preprocess"]
model_final = pipe_final_best.named_steps["model"]

feature_names = preprocessor.get_feature_names_out()
coefs = model_final.coef_

feat_importance = (
    pd.DataFrame({"feature": feature_names, "importance": np.abs(coefs)})
    .sort_values("importance", ascending=False)
    .head(15)
)

fig_importance = px.bar(
    feat_importance,
    x="importance",
    y="feature",
    orientation="h",
    title="Final Model Feature Importance (|Ridge Coefficients|)",
    labels={"importance": "|Coefficient|", "feature": "Feature"},
)
fig_importance.update_layout(yaxis=dict(categoryorder="total ascending"))
fig_importance


In [97]:
# Export key Plotly figures to standalone HTML files for the website.
# This cell assumes all earlier analysis cells have been run so the
# corresponding figure objects exist in memory.

from pathlib import Path
import plotly.io as pio

# Create root-level assets directory (one level above this notebook)
assets_dir = Path("../assets")
assets_dir.mkdir(exist_ok=True)

# Step 2 – EDA plots
pio.write_html(fig_region,          assets_dir / "step2_region.html",          include_plotlyjs="cdn", full_html=True)
pio.write_html(fig_customers,       assets_dir / "step2_customers.html",       include_plotlyjs="cdn", full_html=True)
pio.write_html(fig_box,             assets_dir / "step2_box.html",             include_plotlyjs="cdn", full_html=True)
pio.write_html(fig_scatter,         assets_dir / "step2_scatter.html",         include_plotlyjs="cdn", full_html=True)

# Step 3 – Missingness plot
pio.write_html(fig_miss,            assets_dir / "step3_missingness.html",     include_plotlyjs="cdn", full_html=True)

# Step 6 – Baseline model plots
pio.write_html(fig_baseline_scatter,assets_dir / "step6_baseline_scatter.html",include_plotlyjs="cdn", full_html=True)
pio.write_html(fig_baseline_resid,  assets_dir / "step6_baseline_resid.html",  include_plotlyjs="cdn", full_html=True)

# Step 7 – Final model plots
pio.write_html(fig_final_scatter,   assets_dir / "step7_final_scatter.html",   include_plotlyjs="cdn", full_html=True)
pio.write_html(fig_final_resid,     assets_dir / "step7_final_resid.html",     include_plotlyjs="cdn", full_html=True)
pio.write_html(fig_importance,      assets_dir / "step7_importance.html",      include_plotlyjs="cdn", full_html=True)